# Enterprise Supply Chain Data Generator

Generates **10-15 GB** of realistic multi-tier supply chain data.

**Features:**
- 3 years of data (2022-2024)
- 200 stores, 1000 SKUs, 100 suppliers
- Chunked generation (handles memory limits)
- Parquet format (3-4x compression)

**Runtime:** ~30-45 minutes

In [ ]:
# Step 1: Mount Google Drive & Install Dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install pyarrow fastparquet -q
print("✓ Dependencies installed")

In [ ]:
# Step 2: Configuration
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import gc

np.random.seed(42)

# === OUTPUT DIRECTORY ===
OUTPUT_DIR = '/content/drive/MyDrive/supply_chain_enterprise'

# === ENTERPRISE SCALE CONFIG ===
CONFIG = {
    'n_days': 1095,             # 3 years
    'start_date': '2022-01-01',
    'n_suppliers': 100,
    'n_plants': 8,
    'n_dcs': 25,
    'n_stores': 200,
    'n_skus': 1000,
    'n_categories': 50,
    'n_carriers': 15,
    # Realism settings
    'supplier_delay_prob': 0.15,
    'stockout_threshold': 0.1,
    'missing_data_rate': 0.03,
    'duplicate_rate': 0.01,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Estimate
est_sales_rows = CONFIG['n_days'] * CONFIG['n_stores'] * CONFIG['n_skus']
print(f"Output: {OUTPUT_DIR}")
print(f"Estimated sales rows: {est_sales_rows:,} (~{est_sales_rows * 50 / 1e9:.1f} GB uncompressed)")
print(f"With Parquet compression: ~{est_sales_rows * 15 / 1e9:.1f} GB")

In [ ]:
# Step 3: Master Data Generators

def generate_suppliers(cfg):
    n = cfg['n_suppliers']
    regions = ['APAC', 'EMEA', 'AMER', 'LATAM']
    tiers = ['Strategic', 'Preferred', 'Approved', 'Provisional']
    return pd.DataFrame({
        'supplier_id': [f'SUP_{i:04d}' for i in range(n)],
        'supplier_name': [f'Supplier_{i}' for i in range(n)],
        'region': np.random.choice(regions, n, p=[0.35, 0.25, 0.30, 0.10]),
        'tier': np.random.choice(tiers, n, p=[0.1, 0.25, 0.45, 0.2]),
        'lead_time_days': np.random.randint(7, 60, n),
        'lead_time_std': np.random.randint(1, 10, n),
        'reliability_score': np.round(np.random.beta(8, 2, n), 3),
        'quality_score': np.round(np.random.beta(9, 1, n), 3),
        'unit_cost_multiplier': np.round(np.random.uniform(0.8, 1.3, n), 2),
        'min_order_qty': np.random.choice([100, 250, 500, 1000, 2500], n),
        'payment_terms_days': np.random.choice([30, 45, 60, 90], n),
        'currency': np.random.choice(['USD', 'EUR', 'GBP', 'CNY', 'JPY'], n, p=[0.4, 0.25, 0.1, 0.15, 0.1]),
    })

def generate_products(cfg):
    n = cfg['n_skus']
    categories = [f'CAT_{i:03d}' for i in range(cfg['n_categories'])]
    brands = [f'Brand_{i}' for i in range(20)]
    
    df = pd.DataFrame({
        'sku': [f'SKU_{i:05d}' for i in range(n)],
        'product_name': [f'Product_{i}' for i in range(n)],
        'category': np.random.choice(categories, n),
        'sub_category': [f'SUB_{np.random.randint(0, 5):02d}' for _ in range(n)],
        'brand': np.random.choice(brands, n),
        'unit_cost': np.round(np.random.lognormal(3, 1, n), 2),
        'unit_weight_kg': np.round(np.random.lognormal(0, 1, n), 2),
        'volume_cm3': np.round(np.random.lognormal(6, 1, n), 0),
        'shelf_life_days': np.random.choice([None, 30, 60, 90, 180, 365, 730], n, 
                                            p=[0.3, 0.1, 0.1, 0.15, 0.15, 0.1, 0.1]),
        'is_hazardous': np.random.choice([True, False], n, p=[0.05, 0.95]),
        'is_perishable': np.random.choice([True, False], n, p=[0.2, 0.8]),
        'abc_class': np.random.choice(['A', 'B', 'C'], n, p=[0.2, 0.3, 0.5]),
        'xyz_class': np.random.choice(['X', 'Y', 'Z'], n, p=[0.3, 0.4, 0.3]),
    })
    df['unit_price'] = np.round(df['unit_cost'] * np.random.uniform(1.3, 2.5, n), 2)
    df['safety_stock_days'] = np.where(df['abc_class'] == 'A', 14, 
                                       np.where(df['abc_class'] == 'B', 21, 30))
    return df

def generate_locations(cfg):
    plants = pd.DataFrame({
        'location_id': [f'PLANT_{i:03d}' for i in range(cfg['n_plants'])],
        'location_type': 'PLANT',
        'location_name': [f'Plant_{i}' for i in range(cfg['n_plants'])],
        'region': np.random.choice(['AMER', 'EMEA', 'APAC'], cfg['n_plants']),
        'country': np.random.choice(['USA', 'Germany', 'China', 'Mexico', 'Poland'], cfg['n_plants']),
        'capacity_units_per_day': np.random.randint(10000, 50000, cfg['n_plants']),
        'operating_cost_per_day': np.random.randint(20000, 100000, cfg['n_plants']),
        'latitude': np.random.uniform(25, 55, cfg['n_plants']),
        'longitude': np.random.uniform(-120, 140, cfg['n_plants']),
    })
    dcs = pd.DataFrame({
        'location_id': [f'DC_{i:03d}' for i in range(cfg['n_dcs'])],
        'location_type': 'DC',
        'location_name': [f'DistCenter_{i}' for i in range(cfg['n_dcs'])],
        'region': np.random.choice(['AMER', 'EMEA', 'APAC'], cfg['n_dcs']),
        'country': np.random.choice(['USA', 'Canada', 'UK', 'Germany', 'Japan', 'Australia'], cfg['n_dcs']),
        'capacity_units_per_day': np.random.randint(20000, 100000, cfg['n_dcs']),
        'operating_cost_per_day': np.random.randint(10000, 50000, cfg['n_dcs']),
        'latitude': np.random.uniform(25, 55, cfg['n_dcs']),
        'longitude': np.random.uniform(-120, 140, cfg['n_dcs']),
    })
    stores = pd.DataFrame({
        'location_id': [f'STORE_{i:04d}' for i in range(cfg['n_stores'])],
        'location_type': 'STORE',
        'location_name': [f'Store_{i}' for i in range(cfg['n_stores'])],
        'region': np.random.choice(['AMER', 'EMEA', 'APAC'], cfg['n_stores'], p=[0.5, 0.3, 0.2]),
        'country': np.random.choice(['USA', 'Canada', 'UK', 'Germany', 'France', 'Japan'], cfg['n_stores']),
        'capacity_units_per_day': np.random.randint(1000, 5000, cfg['n_stores']),
        'operating_cost_per_day': np.random.randint(2000, 10000, cfg['n_stores']),
        'latitude': np.random.uniform(25, 55, cfg['n_stores']),
        'longitude': np.random.uniform(-120, 140, cfg['n_stores']),
    })
    return pd.concat([plants, dcs, stores], ignore_index=True)

def generate_carriers(cfg):
    n = cfg['n_carriers']
    return pd.DataFrame({
        'carrier_id': [f'CARR_{i:03d}' for i in range(n)],
        'carrier_name': [f'Carrier_{i}' for i in range(n)],
        'mode': np.random.choice(['TRUCK', 'RAIL', 'AIR', 'OCEAN', 'INTERMODAL'], n, p=[0.4, 0.2, 0.15, 0.15, 0.1]),
        'cost_per_kg_km': np.round(np.random.uniform(0.001, 0.05, n), 4),
        'avg_transit_days': np.random.randint(1, 21, n),
        'on_time_rate': np.round(np.random.beta(9, 1, n), 3),
        'damage_rate': np.round(np.random.beta(1, 50, n), 4),
        'co2_per_kg_km': np.round(np.random.uniform(0.01, 0.5, n), 3),
    })

def generate_supplier_product_mapping(suppliers, products):
    rows = []
    for sku in products['sku']:
        n_suppliers = np.random.randint(1, 5)
        selected = np.random.choice(suppliers['supplier_id'], n_suppliers, replace=False)
        for i, sup in enumerate(selected):
            rows.append({
                'sku': sku, 
                'supplier_id': sup, 
                'is_primary': i == 0, 
                'supplier_sku': f'{sup}_{sku}',
                'moq': np.random.choice([50, 100, 250, 500]),
            })
    return pd.DataFrame(rows)

print("✓ Master data generators loaded")

In [ ]:
# Step 4: Chunked Sales Generator (Memory Efficient)

def generate_sales_chunk(cfg, products, stores, start_date, end_date, chunk_id):
    """Generate sales for a date range (1 month typically)"""
    dates = pd.date_range(start_date, end_date)
    skus = products['sku'].values
    
    grid = pd.MultiIndex.from_product([dates, stores, skus], names=['date', 'store_id', 'sku'])
    df = pd.DataFrame(index=grid).reset_index()
    
    # ABC class mapping
    abc_map = products.set_index('sku')['abc_class'].to_dict()
    df['abc_class'] = df['sku'].map(abc_map)
    base_demand = {'A': 50, 'B': 20, 'C': 8}
    df['base_demand'] = df['abc_class'].map(base_demand)
    
    # Seasonality
    df['day_of_year'] = df['date'].dt.dayofyear
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['seasonality'] = (
        1 + 
        0.3 * np.sin(2 * np.pi * df['day_of_year'] / 365) +  # yearly
        0.1 * np.sin(2 * np.pi * df['day_of_week'] / 7) +    # weekly
        0.15 * np.where(df['month'].isin([11, 12]), 1, 0)    # Q4 boost
    )
    
    # Holiday spikes
    year = start_date.year
    holiday_dates = []
    for y in [year, year + 1]:
        holiday_dates.extend([
            f'{y}-01-01', f'{y}-02-14', f'{y}-07-04', 
            f'{y}-11-24', f'{y}-11-25', f'{y}-12-24', f'{y}-12-25', f'{y}-12-26'
        ])
    holiday_dates = pd.to_datetime(holiday_dates)
    holiday_range = []
    for hd in holiday_dates:
        holiday_range.extend(pd.date_range(start=hd - timedelta(5), end=hd + timedelta(2)).tolist())
    df['is_holiday'] = df['date'].isin(holiday_range)
    df['holiday_mult'] = np.where(df['is_holiday'], np.random.uniform(1.5, 3.0, len(df)), 1.0)
    
    # Promotions
    df['is_promo'] = np.random.random(len(df)) < 0.12
    df['promo_lift'] = np.where(df['is_promo'], np.random.uniform(1.2, 2.0, len(df)), 1.0)
    
    # Trend
    start_ref = pd.Timestamp(cfg['start_date'])
    df['trend'] = 1 + 0.0002 * (df['date'] - start_ref).dt.days
    
    # Compute demand
    n = len(df)
    df['demand'] = (df['base_demand'] * df['seasonality'] * df['holiday_mult'] * df['promo_lift'] * df['trend'])
    df['demand'] = np.maximum(0, np.random.poisson(df['demand'].values))
    
    # Stockouts
    stockout_mask = np.random.random(n) < cfg['stockout_threshold']
    df['units_sold'] = np.where(stockout_mask, (df['demand'] * np.random.uniform(0.3, 0.8, n)).astype(int), df['demand'])
    df['stockout_flag'] = stockout_mask & (df['demand'] > df['units_sold'])
    df['lost_sales'] = df['demand'] - df['units_sold']
    
    # Revenue
    price_map = products.set_index('sku')['unit_price'].to_dict()
    cost_map = products.set_index('sku')['unit_cost'].to_dict()
    df['unit_price'] = df['sku'].map(price_map)
    df['unit_cost'] = df['sku'].map(cost_map)
    df['discount_pct'] = np.where(df['is_promo'], np.random.uniform(0.10, 0.25, n), 0)
    df['revenue'] = np.round(df['units_sold'] * df['unit_price'] * (1 - df['discount_pct']), 2)
    df['cogs'] = np.round(df['units_sold'] * df['unit_cost'], 2)
    df['gross_profit'] = df['revenue'] - df['cogs']
    
    # Select columns
    cols = ['date', 'store_id', 'sku', 'units_sold', 'demand', 'lost_sales',
            'revenue', 'cogs', 'gross_profit', 'discount_pct', 'is_promo', 'stockout_flag']
    
    return df[cols]

print("✓ Chunked sales generator loaded")

In [ ]:
# Step 5: Chunked Inventory Generator

def generate_inventory_chunk(cfg, products, locations, start_date, end_date):
    """Generate weekly inventory snapshots for a date range"""
    dates = pd.date_range(start_date, end_date, freq='W')
    all_locations = locations['location_id'].values
    skus = products['sku'].values
    
    grid = pd.MultiIndex.from_product([dates, all_locations, skus], 
                                       names=['snapshot_date', 'location_id', 'sku'])
    df = pd.DataFrame(index=grid).reset_index()
    
    # Filter: stores get 60% of SKUs
    loc_type_map = locations.set_index('location_id')['location_type'].to_dict()
    df['loc_type'] = df['location_id'].map(loc_type_map)
    store_mask = df['loc_type'] == 'STORE'
    keep_mask = ~store_mask | (np.random.random(len(df)) < 0.6)
    df = df[keep_mask].reset_index(drop=True)
    
    # Inventory levels
    abc_map = products.set_index('sku')['abc_class'].to_dict()
    cost_map = products.set_index('sku')['unit_cost'].to_dict()
    
    df['abc_class'] = df['sku'].map(abc_map)
    base_map = {'A': 500, 'B': 200, 'C': 100}
    df['base'] = df['abc_class'].map(base_map)
    
    # Scale by location type
    loc_scale = {'PLANT': 5, 'DC': 3, 'STORE': 1}
    df['loc_scale'] = df['loc_type'].map(loc_scale)
    df['base'] = df['base'] * df['loc_scale']
    
    n = len(df)
    df['on_hand_qty'] = np.maximum(0, (df['base'] * np.random.lognormal(0, 0.5, n)).astype(int))
    df['in_transit_qty'] = np.maximum(0, (df['base'] * 0.3 * np.random.random(n)).astype(int))
    df['allocated_qty'] = np.maximum(0, (df['base'] * 0.2 * np.random.random(n)).astype(int))
    df['backorder_qty'] = np.where(np.random.random(n) < 0.08, 
                                    (df['base'] * 0.1 * np.random.random(n)).astype(int), 0)
    df['unit_cost'] = df['sku'].map(cost_map)
    df['inventory_value'] = np.round(df['on_hand_qty'] * df['unit_cost'], 2)
    
    # Days of supply (random)
    df['days_of_supply'] = np.round(np.random.lognormal(2.5, 0.5, n), 1)
    
    cols = ['snapshot_date', 'location_id', 'sku', 'on_hand_qty', 'in_transit_qty',
            'allocated_qty', 'backorder_qty', 'inventory_value', 'days_of_supply']
    
    return df[cols]

print("✓ Chunked inventory generator loaded")

In [ ]:
# Step 6: Other Transactional Generators

def generate_demand_forecasts_chunk(sales_chunk, products):
    """Generate forecasts from a sales chunk"""
    sales = sales_chunk.copy()
    sales['week'] = sales['date'].dt.to_period('W').dt.start_time
    weekly = sales.groupby(['week', 'store_id', 'sku']).agg({
        'units_sold': 'sum', 
        'demand': 'sum',
        'revenue': 'sum'
    }).reset_index()
    
    abc_map = products.set_index('sku')['abc_class'].to_dict()
    weekly['abc_class'] = weekly['sku'].map(abc_map)
    error_std = {'A': 0.10, 'B': 0.15, 'C': 0.22}
    weekly['error_std'] = weekly['abc_class'].map(error_std)
    
    n = len(weekly)
    weekly['forecast_error'] = np.random.normal(0, weekly['error_std'])
    weekly['forecast_qty'] = np.maximum(0, np.round(weekly['demand'] * (1 + weekly['forecast_error'])).astype(int))
    weekly['actual_qty'] = weekly['demand']
    weekly['forecast_revenue'] = np.round(weekly['revenue'] * (1 + weekly['forecast_error']), 2)
    weekly['mape'] = np.abs(weekly['forecast_error'])
    
    return weekly[['week', 'store_id', 'sku', 'forecast_qty', 'actual_qty', 
                   'forecast_revenue', 'mape']].rename(columns={'week': 'forecast_date'})

def generate_purchase_orders(cfg, suppliers, products, supplier_product_map):
    """Generate all purchase orders"""
    months = pd.date_range(cfg['start_date'], periods=cfg['n_days']//30, freq='MS')
    skus = products['sku'].values
    
    rows = []
    po_id = 0
    
    for month in months:
        # Each SKU gets 0-2 POs per month
        for sku in skus:
            if np.random.random() > 0.7:  # 70% chance of PO
                continue
                
            supplier_row = supplier_product_map[
                (supplier_product_map['sku'] == sku) & (supplier_product_map['is_primary'])
            ]
            if len(supplier_row) == 0:
                continue
            supplier_id = supplier_row.iloc[0]['supplier_id']
            sup_data = suppliers[suppliers['supplier_id'] == supplier_id].iloc[0]
            
            lead_time = int(sup_data['lead_time_days'])
            min_qty = int(sup_data['min_order_qty'])
            order_qty = np.random.randint(1, 5) * min_qty
            order_date = month + timedelta(days=np.random.randint(0, 15))
            expected_date = order_date + timedelta(days=lead_time)
            
            # Delays
            if np.random.random() > float(sup_data['reliability_score']):
                delay = int(np.random.exponential(lead_time * 0.3))
            else:
                delay = max(0, int(np.random.normal(0, int(sup_data['lead_time_std']))))
            actual_date = expected_date + timedelta(days=delay)
            
            status = 'DELIVERED' if actual_date <= datetime.now() else (
                'IN_TRANSIT' if order_date <= datetime.now() else 'PLANNED')
            
            unit_cost = float(products[products['sku'] == sku]['unit_cost'].iloc[0] * 
                             sup_data['unit_cost_multiplier'])
            
            rows.append({
                'po_id': f'PO_{po_id:08d}',
                'supplier_id': supplier_id,
                'sku': sku,
                'order_qty': order_qty,
                'unit_cost': round(unit_cost, 2),
                'total_cost': round(unit_cost * order_qty, 2),
                'order_date': order_date,
                'expected_date': expected_date,
                'actual_date': actual_date if status == 'DELIVERED' else None,
                'status': status,
                'delay_days': delay if status == 'DELIVERED' else None,
                'currency': sup_data['currency'],
            })
            po_id += 1
    
    return pd.DataFrame(rows)

def generate_shipments(cfg, locations, carriers, purchase_orders):
    """Generate shipments"""
    plants = locations[locations['location_type'] == 'PLANT']['location_id'].values
    dcs = locations[locations['location_type'] == 'DC']['location_id'].values
    stores = locations[locations['location_type'] == 'STORE']['location_id'].values
    carrier_ids = carriers['carrier_id'].values
    
    rows = []
    shipment_id = 0
    
    # From POs
    for _, po in purchase_orders[purchase_orders['status'] == 'DELIVERED'].iterrows():
        carrier = np.random.choice(carrier_ids)
        carrier_data = carriers[carriers['carrier_id'] == carrier].iloc[0]
        transit = int(carrier_data['avg_transit_days'])
        
        planned = po['expected_date'] - timedelta(days=transit)
        actual = po['actual_date'] - timedelta(days=transit) if po['actual_date'] else None
        
        rows.append({
            'shipment_id': f'SHIP_{shipment_id:08d}',
            'origin': po['supplier_id'],
            'destination': np.random.choice(plants),
            'carrier_id': carrier,
            'sku': po['sku'],
            'qty': po['order_qty'],
            'weight_kg': round(po['order_qty'] * np.random.uniform(0.5, 5), 2),
            'planned_ship_date': planned,
            'actual_ship_date': actual,
            'planned_delivery_date': po['expected_date'],
            'actual_delivery_date': po['actual_date'],
            'freight_cost': round(po['order_qty'] * float(carrier_data['cost_per_kg_km']) * np.random.uniform(100, 1000), 2),
            'status': 'DELIVERED',
            'shipment_type': 'INBOUND',
        })
        shipment_id += 1
    
    # Internal shipments (Plant→DC, DC→Store)
    dates = pd.date_range(cfg['start_date'], periods=cfg['n_days'], freq='W')
    for date in dates:
        # Plant to DC
        for _ in range(np.random.randint(10, 30)):
            carrier = np.random.choice(carrier_ids)
            carrier_data = carriers[carriers['carrier_id'] == carrier].iloc[0]
            transit = int(carrier_data['avg_transit_days'])
            
            rows.append({
                'shipment_id': f'SHIP_{shipment_id:08d}',
                'origin': np.random.choice(plants),
                'destination': np.random.choice(dcs),
                'carrier_id': carrier,
                'sku': None,
                'qty': np.random.randint(1000, 15000),
                'weight_kg': round(np.random.uniform(1000, 10000), 2),
                'planned_ship_date': date,
                'actual_ship_date': date + timedelta(days=np.random.randint(-1, 2)),
                'planned_delivery_date': date + timedelta(days=transit),
                'actual_delivery_date': date + timedelta(days=transit + np.random.randint(-2, 5)),
                'freight_cost': round(np.random.uniform(1000, 10000), 2),
                'status': 'DELIVERED' if date < datetime.now() - timedelta(days=transit) else 'IN_TRANSIT',
                'shipment_type': 'TRANSFER',
            })
            shipment_id += 1
        
        # DC to Store
        for _ in range(np.random.randint(50, 150)):
            carrier = np.random.choice(carrier_ids)
            carrier_data = carriers[carriers['carrier_id'] == carrier].iloc[0]
            transit = max(1, int(carrier_data['avg_transit_days']) // 2)
            
            rows.append({
                'shipment_id': f'SHIP_{shipment_id:08d}',
                'origin': np.random.choice(dcs),
                'destination': np.random.choice(stores),
                'carrier_id': carrier,
                'sku': None,
                'qty': np.random.randint(100, 2000),
                'weight_kg': round(np.random.uniform(100, 1000), 2),
                'planned_ship_date': date,
                'actual_ship_date': date + timedelta(days=np.random.randint(0, 2)),
                'planned_delivery_date': date + timedelta(days=transit),
                'actual_delivery_date': date + timedelta(days=transit + np.random.randint(-1, 3)),
                'freight_cost': round(np.random.uniform(100, 1000), 2),
                'status': 'DELIVERED' if date < datetime.now() - timedelta(days=transit) else 'IN_TRANSIT',
                'shipment_type': 'OUTBOUND',
            })
            shipment_id += 1
    
    return pd.DataFrame(rows)

def generate_production_orders(cfg, products, locations):
    """Generate production orders"""
    plants = locations[locations['location_type'] == 'PLANT']['location_id'].values
    skus = products['sku'].values
    dates = pd.date_range(cfg['start_date'], periods=cfg['n_days'])
    
    rows = []
    prod_id = 0
    
    for date in dates:
        for plant in plants:
            for _ in range(np.random.randint(10, 30)):
                sku = np.random.choice(skus)
                planned_qty = np.random.randint(500, 5000)
                yield_rate = np.random.beta(25, 1)  # ~96% avg yield
                actual_qty = int(planned_qty * yield_rate)
                status = np.random.choice(['COMPLETED', 'IN_PROGRESS', 'PLANNED'], p=[0.7, 0.2, 0.1])
                
                rows.append({
                    'production_order_id': f'PROD_{prod_id:08d}',
                    'plant_id': plant,
                    'sku': sku,
                    'planned_qty': planned_qty,
                    'actual_qty': actual_qty if status == 'COMPLETED' else None,
                    'scrap_qty': planned_qty - actual_qty if status == 'COMPLETED' else None,
                    'planned_date': date,
                    'completion_date': date if status == 'COMPLETED' else None,
                    'yield_rate': round(yield_rate, 4) if status == 'COMPLETED' else None,
                    'status': status,
                    'shift': np.random.choice(['DAY', 'EVENING', 'NIGHT']),
                })
                prod_id += 1
    
    return pd.DataFrame(rows)

def inject_data_quality_issues(df, cfg):
    """Add realistic data quality problems"""
    df = df.copy()
    n = len(df)
    
    if cfg['missing_data_rate'] > 0:
        non_key_cols = [c for c in df.columns if not c.endswith('_id') and c != 'sku' and 'date' not in c]
        for col in non_key_cols:
            mask = np.random.random(n) < cfg['missing_data_rate']
            df.loc[mask, col] = None
    
    if cfg['duplicate_rate'] > 0 and n > 100:
        n_dups = int(n * cfg['duplicate_rate'])
        dup_indices = np.random.choice(df.index, n_dups, replace=False)
        df = pd.concat([df, df.loc[dup_indices].copy()], ignore_index=True)
    
    return df

print("✓ Transactional generators loaded")

In [ ]:
# Step 7: Main Generation (Chunked)

def generate_all_data_chunked(cfg, output_dir):
    """Generate all data in memory-efficient chunks"""
    
    print("="*60)
    print("ENTERPRISE SUPPLY CHAIN DATA GENERATOR")
    print("="*60)
    
    # === MASTER DATA ===
    print("\n[1/7] Generating master data...")
    suppliers = generate_suppliers(cfg)
    products = generate_products(cfg)
    locations = generate_locations(cfg)
    carriers = generate_carriers(cfg)
    supplier_product_map = generate_supplier_product_mapping(suppliers, products)
    
    # Save master data
    suppliers.to_parquet(f'{output_dir}/suppliers.parquet', index=False)
    products.to_parquet(f'{output_dir}/products.parquet', index=False)
    locations.to_parquet(f'{output_dir}/locations.parquet', index=False)
    carriers.to_parquet(f'{output_dir}/carriers.parquet', index=False)
    supplier_product_map.to_parquet(f'{output_dir}/supplier_product_map.parquet', index=False)
    print(f"  ✓ Master data saved")
    
    stores = locations[locations['location_type'] == 'STORE']['location_id'].values
    
    # === SALES (CHUNKED BY MONTH) ===
    print("\n[2/7] Generating sales data (chunked by month)...")
    sales_dir = f'{output_dir}/sales'
    os.makedirs(sales_dir, exist_ok=True)
    
    start = pd.Timestamp(cfg['start_date'])
    months = pd.date_range(start, periods=cfg['n_days']//30 + 1, freq='MS')
    
    total_sales_rows = 0
    for i, month_start in enumerate(months[:-1]):
        month_end = months[i + 1] - timedelta(days=1)
        
        sales_chunk = generate_sales_chunk(cfg, products, stores, month_start, month_end, i)
        sales_chunk.to_parquet(f'{sales_dir}/sales_{month_start.strftime("%Y%m")}.parquet', index=False)
        
        total_sales_rows += len(sales_chunk)
        print(f"  ✓ {month_start.strftime('%Y-%m')}: {len(sales_chunk):,} rows")
        
        del sales_chunk
        gc.collect()
    
    print(f"  Total sales rows: {total_sales_rows:,}")
    
    # === DEMAND FORECASTS (CHUNKED) ===
    print("\n[3/7] Generating demand forecasts...")
    forecasts_dir = f'{output_dir}/demand_forecasts'
    os.makedirs(forecasts_dir, exist_ok=True)
    
    total_forecast_rows = 0
    for i, month_start in enumerate(months[:-1]):
        month_end = months[i + 1] - timedelta(days=1)
        
        # Reload sales chunk
        sales_chunk = pd.read_parquet(f'{sales_dir}/sales_{month_start.strftime("%Y%m")}.parquet')
        forecast_chunk = generate_demand_forecasts_chunk(sales_chunk, products)
        forecast_chunk.to_parquet(f'{forecasts_dir}/forecasts_{month_start.strftime("%Y%m")}.parquet', index=False)
        
        total_forecast_rows += len(forecast_chunk)
        
        del sales_chunk, forecast_chunk
        gc.collect()
    
    print(f"  ✓ Total forecast rows: {total_forecast_rows:,}")
    
    # === INVENTORY (CHUNKED BY QUARTER) ===
    print("\n[4/7] Generating inventory snapshots...")
    inventory_dir = f'{output_dir}/inventory'
    os.makedirs(inventory_dir, exist_ok=True)
    
    quarters = pd.date_range(start, periods=cfg['n_days']//90 + 1, freq='QS')
    total_inv_rows = 0
    
    for i, q_start in enumerate(quarters[:-1]):
        q_end = quarters[i + 1] - timedelta(days=1)
        
        inv_chunk = generate_inventory_chunk(cfg, products, locations, q_start, q_end)
        inv_chunk.to_parquet(f'{inventory_dir}/inventory_{q_start.strftime("%Y")}Q{(q_start.month-1)//3 + 1}.parquet', index=False)
        
        total_inv_rows += len(inv_chunk)
        print(f"  ✓ {q_start.strftime('%Y')} Q{(q_start.month-1)//3 + 1}: {len(inv_chunk):,} rows")
        
        del inv_chunk
        gc.collect()
    
    print(f"  Total inventory rows: {total_inv_rows:,}")
    
    # === PURCHASE ORDERS ===
    print("\n[5/7] Generating purchase orders...")
    purchase_orders = generate_purchase_orders(cfg, suppliers, products, supplier_product_map)
    purchase_orders = inject_data_quality_issues(purchase_orders, cfg)
    purchase_orders.to_parquet(f'{output_dir}/purchase_orders.parquet', index=False)
    print(f"  ✓ {len(purchase_orders):,} rows")
    
    # === SHIPMENTS ===
    print("\n[6/7] Generating shipments...")
    shipments = generate_shipments(cfg, locations, carriers, purchase_orders)
    shipments = inject_data_quality_issues(shipments, cfg)
    shipments.to_parquet(f'{output_dir}/shipments.parquet', index=False)
    print(f"  ✓ {len(shipments):,} rows")
    
    del purchase_orders, shipments
    gc.collect()
    
    # === PRODUCTION ORDERS ===
    print("\n[7/7] Generating production orders...")
    production = generate_production_orders(cfg, products, locations)
    production = inject_data_quality_issues(production, cfg)
    production.to_parquet(f'{output_dir}/production_orders.parquet', index=False)
    print(f"  ✓ {len(production):,} rows")
    
    del production
    gc.collect()
    
    # === SUMMARY ===
    print("\n" + "="*60)
    print("✅ DATA GENERATION COMPLETE")
    print("="*60)
    print(f"📁 Location: {output_dir}")
    print(f"📊 Total sales rows: {total_sales_rows:,}")
    print(f"📊 Total forecast rows: {total_forecast_rows:,}")
    print(f"📊 Total inventory rows: {total_inv_rows:,}")
    
    return output_dir

print("✓ Main generator loaded")

In [ ]:
# Step 8: RUN THE GENERATOR
# This will take ~30-45 minutes

generate_all_data_chunked(CONFIG, OUTPUT_DIR)

In [ ]:
# Step 9: Verify Output
import os

def get_size(path):
    total = 0
    if os.path.isfile(path):
        return os.path.getsize(path)
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

print("Dataset Summary:")
print("-" * 50)
total_size = 0

for item in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, item)
    size = get_size(path)
    total_size += size
    
    if os.path.isdir(path):
        n_files = len(os.listdir(path))
        print(f"  {item}/ ({n_files} files): {size / 1e9:.2f} GB")
    else:
        print(f"  {item}: {size / 1e6:.1f} MB")

print("-" * 50)
print(f"TOTAL: {total_size / 1e9:.2f} GB")

In [ ]:
# Step 10: Test Query with DuckDB
!pip install duckdb -q
import duckdb

con = duckdb.connect()

# Query partitioned sales data
print("Top 10 SKUs by revenue (querying all partitions):")
result = con.execute(f"""
    SELECT 
        sku,
        SUM(revenue) as total_revenue,
        SUM(units_sold) as total_units,
        COUNT(*) as num_transactions
    FROM '{OUTPUT_DIR}/sales/*.parquet'
    GROUP BY sku
    ORDER BY total_revenue DESC
    LIMIT 10
""").df()

display(result)

In [ ]:
# Bonus: Monthly revenue trend
print("\nMonthly Revenue Trend:")
trend = con.execute(f"""
    SELECT 
        DATE_TRUNC('month', date) as month,
        SUM(revenue) as revenue,
        SUM(units_sold) as units,
        SUM(CASE WHEN stockout_flag THEN 1 ELSE 0 END) as stockout_events
    FROM '{OUTPUT_DIR}/sales/*.parquet'
    GROUP BY 1
    ORDER BY 1
""").df()

display(trend)